# 02 — Arabic Text Cleaning

**Goal:** Load the raw labeled and unlabeled complaint files, apply Arabic text cleaning/normalization, and save two cleaned CSVs to the shared Google Drive.

**Inputs (from repo):**
- `data/text/complaints_labeled.csv`
- `data/text/complaints_unlabeled.csv`

**Outputs (saved to Drive):**
- `NLP-Complaints-Team/data/processed/complaints_labeled_clean.csv`
- `NLP-Complaints-Team/data/processed/complaints_unlabeled_clean.csv`

> **Before running:** make sure you have already run `00_setup.ipynb` this session to mount Drive and pull the repo.

In [ ]:
# Cell 2 — Mount Drive + pull repo
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/drive/MyDrive/NLP-Complaints-Team/repo'
os.system(f'git -C "{REPO_DIR}" pull')
print('Repo updated.')

In [ ]:
# Cell 3 — Install dependencies
import subprocess, sys

packages = [
    'scikit-learn>=1.3',
    'camel-tools>=1.5',
    'pandas>=2.0',
    'numpy>=1.24',
    'joblib>=1.3',
    'fastapi>=0.110',
    'uvicorn>=0.29',
    'gradio>=4.0',
    'seaborn>=0.13',
    'matplotlib>=3.8',
]

subprocess.check_call(
    [sys.executable, '-m', 'pip', 'install', '--quiet'] + packages
)
print('All packages installed.')

In [ ]:
# Cell 4 — Load data
import pandas as pd

REPO_DIR = '/content/drive/MyDrive/NLP-Complaints-Team/repo'

labeled   = pd.read_csv(f'{REPO_DIR}/data/text/complaints_labeled.csv')
unlabeled = pd.read_csv(f'{REPO_DIR}/data/text/complaints_unlabeled.csv')

print(f'Labeled rows:   {len(labeled)}')
print(f'Unlabeled rows: {len(unlabeled)}')
print(labeled.head(3))

In [ ]:
# Cell 5 — Arabic cleaning function
import re
from camel_tools.utils.normalize import normalize_unicode, normalize_alef_ar
from camel_tools.utils.dediac import dediac_ar

def clean_arabic(text: str) -> str:
    text = normalize_unicode(text)       # fix encoding variants
    text = dediac_ar(text)               # remove tashkeel (diacritics)
    text = normalize_alef_ar(text)       # أ إ آ → ا
    text = re.sub('[ىی]', 'ي', text)     # alef maqsura → ya
    text = re.sub('ة', 'ه', text)        # ta marbuta → ha
    text = re.sub(r'http\S+|www\S+', '', text)              # remove URLs
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)          # keep Arabic + spaces only
    text = re.sub(r'\s+', ' ', text).strip()                # collapse spaces
    return text

# Quick test
print(clean_arabic('\u062a\u0648\u0627\u0635\u0644\u062a\u064f \u0645\u0639 \u062e\u062f\u0645\u0629 \u0627\u0644\u0639\u0645\u0644\u0627\u0621 \u0639\u062f\u0629 \u0645\u0631\u0627\u062a'))

In [ ]:
# Cell 6 — Apply cleaning + save
OUT_DIR = '/content/drive/MyDrive/NLP-Complaints-Team/data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

labeled['text_clean']   = labeled['text'].apply(clean_arabic)
unlabeled['text_clean'] = unlabeled['text'].apply(clean_arabic)

labeled.to_csv(  f'{OUT_DIR}/complaints_labeled_clean.csv',   index=False)
unlabeled.to_csv(f'{OUT_DIR}/complaints_unlabeled_clean.csv', index=False)

print('Saved:')
print(f'  {OUT_DIR}/complaints_labeled_clean.csv   ({len(labeled)} rows)')
print(f'  {OUT_DIR}/complaints_unlabeled_clean.csv ({len(unlabeled)} rows)')

In [ ]:
# Cell 7 — Spot-check: 5 side-by-side original vs cleaned rows
sample = labeled[['text', 'text_clean']].sample(5, random_state=1).reset_index(drop=True)
sample.columns = ['original', 'cleaned']

pd.set_option('display.max_colwidth', 80)
print(sample.to_string(index=False))

## Cell 8 — Commit notebook to git

After verifying the outputs above, commit and push **only the notebook** on your branch:

```bash
git -C /content/drive/MyDrive/NLP-Complaints-Team/repo add notebooks/02_text_cleaning.ipynb
git -C /content/drive/MyDrive/NLP-Complaints-Team/repo commit -m "Add text cleaning notebook"
git -C /content/drive/MyDrive/NLP-Complaints-Team/repo push origin lana/data-processing
```

> **Do NOT commit the CSV files** — they live only on the shared Drive.